# The joint model: warmup, fade, anchor
One encoder, one predictor, one decoder (`../docs/findings.md`, section 10 and the design discussion after it):

* **Warmup** (`joint` arm): class-weighted reconstruction keeps the content available, the prediction gradient reaches the encoder and shapes the
  latent for dynamics, SIGReg keeps it healthy. The weights are the ground-truth tile classes with a **softer cap (4)**, so terrain (weight 0.22
  instead of 0.04) and mob (1.04 instead of 0.19) are not zeroed (the hand's prerequisite is the terrain). Ground truth is used on purpose: it
  isolates the architecture and the schedule from the choice of weights (later: per-tile error weights).
* **Fade**: the reconstruction weight falls to a small floor; prediction then prunes what was not learned.
* **Anchor**: prediction distillation. The live (encoder, predictor) is held to the predictions of a frozen snapshot of itself. It holds exactly what
  the snapshot could predict (the learned part) and leaves noise and unlearned content free.

Only the comparisons that are needed, one seed:

| | run | question |
|---|---|---|
| Stage 1 | **joint** and the **shadow alone (`ae`)**, same weights, same FiLM predictor, 20k steps | does the prediction gradient make the latent learnable for dynamics (the shadow's predictor on a reconstruction latent barely foresaw the facing: 0.36-0.46 decoded accuracy on move steps, ceiling 0.96)? **Stop condition: if not, nothing below is worth building on.** |
| Stage 2 | four **forks** of the joint checkpoint (no retraining of the warmup): `continue`, `fade`, `anchor`, `freeze` | does the fade release noise, does it erode what was learned, and does the anchor stop that? `freeze` is V-JEPA 2's order (dynamics on a frozen encoder) |

Everything is read **with decoders** (`gaf/readout.py`), never with small probes: they under-read compressed latents.

In [ ]:
%pip install -q -e ..
import torch
print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU (CPU fallback, slow)")

In [ ]:
%matplotlib inline
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from gaf.env_grid import GridCfg
from gaf.train import Cfg
from gaf import analysis as A
from gaf import readout as R
from gaf import fork as F

OUT = "/workspace/results/joint"
SEED, D, STEPS, WORKERS = 0, 128, 20_000, 2
FORK_STEPS, FADE, R_FLOOR, BETA = 6000, 3000, 0.1, 10.0   # stage 2: 3000 steps of fade, then 3000 at the floor; beta = weight of the anchor
DS_STEPS = 3000                                           # steps of the fresh decoder of the decoder-strength readout (the same for every row)
ENV = GridCfg(W=12, H=8, hand_tiles=1, agent_shape="arrow", mob_mode="chase", n_mobs=1, mob_chase=0.7)
COMMON = dict(lam=0.01, batch=256, lr=1e-3, wd=0.01, steps=STEPS, floor_start=10 ** 9, eval_every=2000, eval_parts=6000, eval_parts_steps=500,
              D=D, seed=SEED, env=ENV, rec_class_power=1.0, rec_class_cap=4.0, pred_arch="film", ckpt_every=5000)
RUNS = {"joint": Cfg(arm="joint", tag="joint-warmup", **COMMON),
        "shadow alone (ae)": Cfg(arm="ae", tag="joint-ae", **COMMON)}
PARENT = Path(OUT) / RUNS["joint"].run_id
pd.set_option("display.precision", 3)

## Stage 1: launch
Background process (safe to re-run: finished and running runs are skipped). To stop: `kill -TERM -- -$(cat /workspace/results/joint/launch_*.pid)`.

In [ ]:
A.launch(list(RUNS.values()), OUT, workers=WORKERS)

In [ ]:
display(A.progress(OUT, list(RUNS.values())))

In [ ]:
A.live_probes(OUT, RUNS, every=30)

**Recovery cell.** Only if a run died in its final probes (it has `ckpt_<steps>.pt` and a full `metrics.csv` but no `summary.json`).

In [ ]:
for name, c in RUNS.items():
    d = Path(OUT) / c.run_id
    if d.exists() and not (d / "summary.json").exists() and (d / f"ckpt_{c.steps}.pt").exists():
        print("finalizing", name)
        A.finalize(c, OUT)

## Stage 1: what each model foresees, and what its latent holds
`dynamics`: the model's own decoder applied to the real next latent (ceiling), to the predictor's output (what it foresees) and to the last frame (floor).
**The column to read first is `facing acc, move steps`**: on steps where the action turned or moved the agent, copying the last frame is mostly wrong and the
ceiling is about 0.96. `decoder strength`: a fresh decoder on the frozen latent (the share of the "decode nothing" error removed; 1 = perfect).
Reference from the B runs (same readouts): the shadow's predictor foresaw facing 0.36-0.46 and the agent 0.49-0.51; a predictor fork on the frozen shadow
reached 0.59 at best.

In [ ]:
dyn1 = {n: R.dynamics(Path(OUT) / c.run_id) for n, c in RUNS.items() if (Path(OUT) / c.run_id / "summary.json").exists()}
content1 = {n: R.decoder_strength(Path(OUT) / c.run_id, steps=DS_STEPS).loc["live"] for n, c in RUNS.items() if n in dyn1}
DYN1, CONTENT1 = pd.concat(dyn1), pd.DataFrame(content1).T
display(DYN1.round(2))
display(CONTENT1.round(2))

**Stage 1 verdict (rules written before the results).**
* *Dynamics shaped:* the joint model's predicted frame beats the shadow alone's by at least 0.15 on move-step facing and 0.10 on the agent. If not, **stop**.
* *Content kept:* the joint latent decodes hand, agent and facing at >= 0.8 (as the shadow does).
* *Hand dynamics* (informative): the predicted hand more than 0.3 above copy-last.

In [ ]:
def g(df, idx, col):
    try:
        return float(df.loc[idx, col])
    except (KeyError, TypeError):
        return float("nan")


def say(label, ok):
    print(f"{label}: {'yes' if ok else 'NO'}")


if len(dyn1) == 2:
    pj, pa = ("live", "predicted"), ("live", "predicted")
    fj, fa = g(dyn1["joint"], pj, "facing acc, move steps"), g(dyn1["shadow alone (ae)"], pa, "facing acc, move steps")
    aj, aa = g(dyn1["joint"], pj, "agent"), g(dyn1["shadow alone (ae)"], pa, "agent")
    print(f"predicted facing on move steps: joint {fj:.2f} vs shadow alone {fa:.2f}; predicted agent: joint {aj:.2f} vs {aa:.2f}")
    say("dynamics shaped (facing +0.15 and agent +0.10)", fj >= fa + 0.15 and aj >= aa + 0.10)
    hold = {p: g(CONTENT1, "joint", p) for p in ("hand", "agent", "facing acc")}
    print("joint latent, decoder strength:", {k: round(v, 2) for k, v in hold.items()})
    say("content kept (hand, agent, facing >= 0.8)", all(v >= 0.8 for v in hold.values()))
    hj, hc = g(dyn1["joint"], pj, "hand"), g(dyn1["joint"], ("live", "copy last frame"), "hand")
    say(f"hand dynamics learned (predicted {hj:.2f} vs copy-last {hc:.2f}, +0.3)", hj > hc + 0.3)
else:
    print("stage 1 not finished")

## Stage 2: the fade and the anchor, as forks of the joint checkpoint
Each fork restarts from the joint checkpoint at its last step and runs 6000 steps (`FADE` steps of fade, then the floor), in its own background process.
`continue` (full reconstruction) is the control, `fade` the fade alone, `anchor` the fade plus prediction distillation, `freeze` trains only the predictor on
the frozen encoder. Start this only if the Stage 1 verdict says the dynamics are shaped (or if you want the answer anyway).

In [ ]:
print(F.launch_forks(PARENT, steps=FORK_STEPS, fade=FADE, r_floor=R_FLOOR, beta=BETA))

In [ ]:
display(F.status(PARENT))

## Stage 2: what the fade releases and what it erodes
The same two readouts for the joint checkpoint (the parent) and for each fork. **Erosion** = the drop of the learned content (agent, inventory, facing) from
the parent; **release** = the drop of twinkle (noise should go); **dynamics kept** = the predicted move-step facing against the parent.

In [ ]:
forks = {m: PARENT / "forks" / m for m in F.MODES if (PARENT / "forks" / m / "done").exists()}
rows = {"parent (joint)": PARENT, **{f"fork: {m}": p for m, p in forks.items()}}
dyn2 = {n: R.dynamics(p) for n, p in rows.items()}
content2 = {n: R.decoder_strength(p, steps=DS_STEPS).loc["live"] for n, p in rows.items()}
CONTENT2 = pd.DataFrame(content2).T
PRED2 = pd.DataFrame({n: d.loc[("live", "predicted")] for n, d in dyn2.items()}).T
display(CONTENT2.round(2))
display(PRED2.round(2))

**Stage 2 verdict (rules written before the results).** Relative to the parent:
* *holds:* none of agent, inventory, facing acc drops by more than 0.05;
* *releases noise:* twinkle drops by at least 0.2;
* *dynamics kept:* the predicted move-step facing drops by at most 0.05.

The anchor works if it holds, releases noise and keeps the dynamics, **and `fade` alone does not** (otherwise the anchor is not needed). `continue` should hold
and not release. `freeze` shows what a frozen encoder gives: holds, does not release.

In [ ]:
if "parent (joint)" in CONTENT2.index and len(CONTENT2) > 1:
    base, basep = CONTENT2.loc["parent (joint)"], PRED2.loc["parent (joint)"]
    for n in CONTENT2.index[1:]:
        drop = {k: base[k] - CONTENT2.loc[n, k] for k in ("agent", "inventory", "facing acc")}
        tw = base["twinkle"] - CONTENT2.loc[n, "twinkle"]
        dyn_drop = basep["facing acc, move steps"] - PRED2.loc[n, "facing acc, move steps"]
        print(f"--- {n}")
        print("  drop of learned content:", {k: round(v, 2) for k, v in drop.items()}, "| twinkle drop:", round(tw, 2), "| move-step facing drop:", round(dyn_drop, 2))
        say("  holds", all(v <= 0.05 for v in drop.values()))
        say("  releases noise", tw >= 0.2)
        say("  dynamics kept", dyn_drop <= 0.05)
else:
    print("stage 2 not finished")

## Download
After the forks are done: zip `/workspace/results/joint` (it holds the checkpoints and `forks/`) and download it, as before.